In [ ]:
# [Célula 1] Setup + localizar arquivos (data/model) e checar colunas disponíveis
import os
from pathlib import Path
import pandas as pd

DATA_CANDIDATES = [
    "datamart_wide_aluno_ano.csv",
    "./datamart_wide_aluno_ano.csv",
    "../data/datamart_wide_aluno_ano.csv",
    "/content/datamart_wide_aluno_ano.csv",
]

MODEL_CANDIDATES = [
    "ian_risk_pipeline.joblib",                 # seu caso (arquivo no root do notebook)
    "./ian_risk_pipeline.joblib",
    "models/ian_risk_pipeline.joblib",
    "../models/ian_risk_pipeline.joblib",
    "/content/ian_risk_pipeline.joblib",
]

DATA_PATH = next((p for p in DATA_CANDIDATES if Path(p).exists()), None)
MODEL_PATH = next((p for p in MODEL_CANDIDATES if Path(p).exists()), None)

assert DATA_PATH is not None, f"❌ Não encontrei o CSV. Procurei em: {DATA_CANDIDATES}"
assert MODEL_PATH is not None, f"❌ Não encontrei o modelo .joblib. Procurei em: {MODEL_CANDIDATES}"

print("✅ DATA_PATH:", Path(DATA_PATH).resolve())
print("✅ MODEL_PATH:", Path(MODEL_PATH).resolve())

# checagem rápida das colunas do CSV (sem inferência ainda)
df_raw = pd.read_csv(DATA_PATH)
print("shape:", df_raw.shape)
print("cols:", df_raw.columns.tolist())
df_raw.head(3)


✅ DATA_PATH: /content/datamart_wide_aluno_ano.csv
✅ MODEL_PATH: /content/ian_risk_pipeline.joblib
shape: (3030, 18)
cols: ['RA', 'NOME', 'FASE', 'TURMA', 'GENERO', 'INSTITUICAO_DE_ENSINO', 'ANO_INGRESSO', 'IAN', 'IDADE', 'IEG', 'IAA', 'IPS', 'IPV', 'INDE', 'PEDRA', 'ANO', 'IPP', 'DEFASAGEM']


,RA,NOME,FASE,TURMA,GENERO,INSTITUICAO_DE_ENSINO,ANO_INGRESSO,IAN,IDADE,IEG,IAA,IPS,IPV,INDE,PEDRA,ANO,IPP,DEFASAGEM
0,RA-1,Aluno-1,7,A,Menina,Escola Pública,2016,5.0,19,4.1,8.3,5.6,7.278,5.783,Quartzo,2022,NaN,NaN
1,RA-2,Aluno-2,7,A,Menina,Rede Decisão,2017,10.0,17,5.2,8.8,6.3,6.778,7.055,Ametista,2022,NaN,NaN
2,RA-3,Aluno-3,7,A,Menina,Rede Decisão,2016,10.0,17,7.9,0.0,5.6,7.556,6.591,Ágata,2022,NaN,NaN


In [ ]:
# [Célula 2] Carregar bundle do modelo e validar estrutura (model + threshold)
import joblib

bundle = joblib.load(MODEL_PATH)
print("Tipo do bundle:", type(bundle))

# o bundle correto é dict com chaves padronizadas
assert isinstance(bundle, dict), f"❌ Bundle deveria ser dict, veio: {type(bundle)}"
print("Keys:", list(bundle.keys()))

required_keys = ["model", "threshold"]
missing_keys = [k for k in required_keys if k not in bundle]
assert not missing_keys, f"❌ Bundle sem chaves obrigatórias: {missing_keys}"

model = bundle["model"]
threshold = float(bundle["threshold"])

best_model_name = bundle.get("best_model_name", "unknown")
target_name = bundle.get("target", "TARGET_RISCO_IAN")
positive_class = bundle.get("positive_class", 1)

print("✅ best_model_name:", best_model_name)
print("✅ threshold:", threshold)
print("✅ target:", target_name)
print("✅ positive_class:", positive_class)
print("✅ model type:", type(model))


Tipo do bundle: <class 'dict'>
Keys: ['model', 'threshold', 'best_model_name', 'target', 'positive_class']
✅ best_model_name: logreg
✅ threshold: 0.6
✅ target: TARGET_RISCO_IAN
✅ positive_class: 1
✅ model type: <class 'sklearn.pipeline.Pipeline'>


In [ ]:
# [Célula 3] Validar colunas obrigatórias e preparar tipos mínimos
import numpy as np
import pandas as pd

df = pd.read_csv(DATA_PATH)

# colunas obrigatórias que o seu modelo estava exigindo (pelo erro)
REQUIRED = ["ANO", "IDADE", "PEDRA", "DEFASAGEM", "ANO_INGRESSO"]

missing = [c for c in REQUIRED if c not in df.columns]
assert not missing, f"❌ Ainda faltam colunas para o modelo: {missing}\nColunas existentes: {df.columns.tolist()}"

# tipos: garante ANO numérico e evita problemas de string com espaços
df["ANO"] = pd.to_numeric(df["ANO"], errors="coerce").astype("Int64")

# se DEFASAGEM vier como -0.0 / NaN, normaliza
if "DEFASAGEM" in df.columns:
    df["DEFASAGEM"] = pd.to_numeric(df["DEFASAGEM"], errors="coerce")

print("✅ OK - colunas obrigatórias presentes.")
print("shape:", df.shape)
df[["RA","NOME","ANO"]].head(3) if all(c in df.columns for c in ["RA","NOME","ANO"]) else df.head(3)


✅ OK - colunas obrigatórias presentes.
shape: (3030, 18)


,RA,NOME,ANO
0,RA-1,Aluno-1,2022
1,RA-2,Aluno-2,2022
2,RA-3,Aluno-3,2022


In [ ]:
# [Célula 4] Diagnóstico do schema esperado + montar df_features compatível (base vs _LAG1)
import pandas as pd

df = pd.read_csv(DATA_PATH)

# tenta descobrir exatamente quais colunas o preprocess espera na entrada
preprocess = None
if hasattr(model, "named_steps") and "preprocess" in model.named_steps:
    preprocess = model.named_steps["preprocess"]

expected_in = None
if preprocess is not None and hasattr(preprocess, "feature_names_in_"):
    expected_in = list(preprocess.feature_names_in_)

print("✅ Colunas no CSV:", sorted(df.columns.tolist()))
print("✅ Modelo tem preprocess?", preprocess is not None)
print("✅ preprocess.feature_names_in_ existe?", expected_in is not None)
if expected_in is not None:
    print("✅ Modelo espera (entrada):", sorted(expected_in))

# --- regra objetiva:
# se o modelo espera *_LAG1, criamos essas colunas a partir das colunas base (sem LAG)
if expected_in is not None and any(c.endswith("_LAG1") for c in expected_in):
    base_needed = [c.replace("_LAG1", "") for c in expected_in]
    missing_base = [c for c in base_needed if c not in df.columns]
    assert not missing_base, (
        "❌ O modelo espera *_LAG1, mas faltam colunas base no CSV para criar isso:\n"
        f"{missing_base}\n\n"
        f"Colunas disponíveis:\n{sorted(df.columns.tolist())}"
    )

    # monta df_features com nomes exatamente como o modelo quer
    rename_map = {c.replace("_LAG1", ""): c for c in expected_in}
    df_features = df[base_needed].rename(columns=rename_map).copy()

    print("✅ df_features montado no formato *_LAG1")
    print("df_features cols:", sorted(df_features.columns.tolist()))

else:
    # caso o modelo não use LAG, segue com df normal
    df_features = df.copy()
    print("✅ df_features = df (sem necessidade de *_LAG1)")


✅ Colunas no CSV: ['ANO', 'ANO_INGRESSO', 'DEFASAGEM', 'FASE', 'GENERO', 'IAA', 'IAN', 'IDADE', 'IEG', 'INDE', 'INSTITUICAO_DE_ENSINO', 'IPP', 'IPS', 'IPV', 'NOME', 'PEDRA', 'RA', 'TURMA']
✅ Modelo tem preprocess? True
✅ preprocess.feature_names_in_ existe? True
✅ Modelo espera (entrada): ['ANO', 'ANO_INGRESSO_LAG1', 'DEFASAGEM_LAG1', 'FASE_LAG1', 'GENERO_LAG1', 'IAA_LAG1', 'IAN_LAG1', 'IDADE_LAG1', 'IEG_LAG1', 'INDE_LAG1', 'INSTITUICAO_DE_ENSINO_LAG1', 'IPP_LAG1', 'IPS_LAG1', 'IPV_LAG1', 'PEDRA_LAG1', 'TURMA_LAG1']
✅ df_features montado no formato *_LAG1
df_features cols: ['ANO', 'ANO_INGRESSO_LAG1', 'DEFASAGEM_LAG1', 'FASE_LAG1', 'GENERO_LAG1', 'IAA_LAG1', 'IAN_LAG1', 'IDADE_LAG1', 'IEG_LAG1', 'INDE_LAG1', 'INSTITUICAO_DE_ENSINO_LAG1', 'IPP_LAG1', 'IPS_LAG1', 'IPV_LAG1', 'PEDRA_LAG1', 'TURMA_LAG1']


In [ ]:
# [Célula X] Sanear tipos (corrigir IDADE_LAG1 como data e INDE_LAG1 como texto) antes do predict
import numpy as np
import pandas as pd

# 1) Corrigir IDADE_LAG1 quando vier como "1900-.."
if "IDADE_LAG1" in df_features.columns:
    s = df_features["IDADE_LAG1"].astype(str)

    mask_1900 = s.str.startswith("1900-")
    if mask_1900.any():
        dt = pd.to_datetime(s.where(mask_1900), errors="coerce")

        # Como seu caso está em 1900-01-XX, o "XX" está representando a idade.
        # (pelo seu print, sempre 1900-01-..)
        df_features.loc[mask_1900, "IDADE_LAG1"] = dt.dt.day.astype(float)

    # força numérico (o que não converter vira NaN e o pipeline imputa)
    df_features["IDADE_LAG1"] = pd.to_numeric(df_features["IDADE_LAG1"], errors="coerce")

# 2) Corrigir INDE_LAG1 quando vier "INCLUIR"
if "INDE_LAG1" in df_features.columns:
    df_features["INDE_LAG1"] = df_features["INDE_LAG1"].replace({"INCLUIR": np.nan})
    df_features["INDE_LAG1"] = pd.to_numeric(df_features["INDE_LAG1"], errors="coerce")

# 3) Garantia extra: tenta converter para numérico todos os *LAG1 que deveriam ser numéricos
num_like = ["IAN_LAG1","IEG_LAG1","IAA_LAG1","IPS_LAG1","IPV_LAG1","IPP_LAG1","DEFASAGEM_LAG1","ANO_INGRESSO_LAG1","ANO"]
for c in num_like:
    if c in df_features.columns:
        df_features[c] = pd.to_numeric(df_features[c], errors="coerce")

print("✅ Sanity types (amostra):")
display(df_features.dtypes)


✅ Sanity types (amostra):


,0
ANO,int64
FASE_LAG1,object
TURMA_LAG1,object
GENERO_LAG1,object
INSTITUICAO_DE_ENSINO_LAG1,object
PEDRA_LAG1,object
ANO_INGRESSO_LAG1,int64
IDADE_LAG1,float64
INDE_LAG1,float64
IEG_LAG1,float64


In [ ]:
# [Célula 5] Rodar inferência (probabilidade + classe usando threshold)
import numpy as np
import pandas as pd

# predict_proba para classe positiva
proba_pos = model.predict_proba(df_features)[:, int(positive_class)]

# aplica threshold do bundle
pred = (proba_pos >= threshold).astype(int)

print("✅ Inferência concluída.")
print("Pred distribution:", pd.Series(pred).value_counts(dropna=False).to_dict())
print("Proba min/mean/max:", float(np.min(proba_pos)), float(np.mean(proba_pos)), float(np.max(proba_pos)))


✅ Inferência concluída.
Pred distribution: {0: 2751, 1: 279}
Proba min/mean/max: 5.802398481980279e-07 0.15751721340927508 0.9709397785472269


In [ ]:
# [Célula 6] Montar output final (IDs + proba + pred + metadados)
import pandas as pd
import numpy as np

out = pd.DataFrame(index=df.index)

# IDs (se existirem)
for c in ID_COLS:
    out[c] = df[c]

out["proba_risco_ian"] = np.round(proba_pos, 6)
out["pred_risco_ian"] = pred.astype(int)

# metadados úteis p/ rastreabilidade
out["threshold_usado"] = threshold
out["modelo_usado"] = best_model_name

print("✅ Output final shape:", out.shape)
display(out.head(10))

# debug: distribuição
print("\nDistribuição pred:")
display(out["pred_risco_ian"].value_counts(normalize=True))


✅ Output final shape: (3030, 7)


,RA,ANO,NOME,proba_risco_ian,pred_risco_ian,threshold_usado,modelo_usado
0,RA-1,2022,Aluno-1,0.377422,0,0.6,logreg
1,RA-2,2022,Aluno-2,0.000431,0,0.6,logreg
2,RA-3,2022,Aluno-3,0.001664,0,0.6,logreg
3,RA-4,2022,Aluno-4,0.000079,0,0.6,logreg
4,RA-5,2022,Aluno-5,0.000948,0,0.6,logreg
5,RA-6,2022,Aluno-6,0.375738,0,0.6,logreg
6,RA-7,2022,Aluno-7,0.617539,1,0.6,logreg
7,RA-8,2022,Aluno-8,0.093492,0,0.6,logreg
8,RA-9,2022,Aluno-9,0.800754,1,0.6,logreg
9,RA-10,2022,Aluno-10,0.425807,0,0.6,logreg



Distribuição pred:


,proportion
pred_risco_ian,
0,0.907921
1,0.092079


In [ ]:
# [Célula 7] Salvar arquivo final (e confirmar que gravou de verdade)
import os
from pathlib import Path

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / "predicoes_risco_ian.csv"
out.to_csv(OUT_PATH, index=False, encoding="utf-8")

print("✅ Salvo em:", OUT_PATH.resolve())
print("✅ Exists:", OUT_PATH.exists())
print("✅ Size (bytes):", OUT_PATH.stat().st_size)


✅ Salvo em: /content/outputs/predicoes_risco_ian.csv
✅ Exists: True
✅ Size (bytes): 113765
